# 🏥 Healthcare Administrative Evaluation with TruLens

Conversational AI assistants are increasingly used in healthcare administrative workflows to help health plan members and providers navigate **claim status updates**, **prior-authorization requirements**, **benefit and coverage explanations**, and **administrative next-step guidance**.

While these assistants do not offer clinical diagnosis or medical treatment, administrative errors (such as misstating prior authorization rules, fabricating reimbursement check dates, or contradicting benefit exclusions) can cause member confusion, financial distress, and workflow escalations.

### Objective
In this notebook, we demonstrate how to measure **Healthcare Groundedness** using TruLens:
1. Evaluating whether an assistant's administrative statements are strictly supported by authoritative plan, policy, or claim context.
2. Detecting **contradictory** or **unsupported/hallucinated** administrative responses.
3. Validating **appropriate abstention** when administrative data or authorization approval is pending or unavailable.

In [ ]:
import json
import os
import pandas as pd

# Load synthetic healthcare administrative test corpus
corpus_path = os.path.join(os.path.dirname(__file__) if '__file__' in globals() else '.', 'healthcare_admin_corpus.jsonl')
if not os.path.exists(corpus_path):
    corpus_path = 'examples/expositional/use_cases/healthcare_admin_corpus.jsonl'

with open(corpus_path, 'r') as f:
    dataset = [json.loads(line) for line in f]

df_dataset = pd.DataFrame(dataset)
print(f"Loaded {len(dataset)} synthetic healthcare administrative evaluation cases.")
df_dataset[['id', 'category', 'scenario', 'query', 'expected_groundedness']].head(8)

## 🔍 Defining the Healthcare Groundedness Metric

We compose TruLens's  and  components to measure groundedness with **Chain-of-Thought (CoT) reasoning**. The metric evaluates each statement in the assistant's response against the retrieved policy or claim context.

In [ ]:
from trulens.core import Metric, Selector

# Setup provider with fallback for offline execution
api_key_present = bool(os.getenv("OPENAI_API_KEY"))

if api_key_present:
    from trulens.providers.openai import OpenAI
    provider = OpenAI(model_engine="gpt-4o-mini")
else:
    print("OPENAI_API_KEY not found. Using simulated mock evaluation provider for demonstration.")
    class MockHealthcareProvider:
        def groundedness_measure_with_cot_reasons(self, source, statement):
            if "approved" in statement.lower() and "denied" in source.lower():
                return 0.0, {"reason": "Statement claims claim was approved, but authoritative source shows DENIED."}
            elif "mailed to your home" in statement.lower() and "eft" in source.lower():
                return 0.0, {"reason": "Statement claims check was mailed, but source states payment was sent via EFT to provider."}
            elif "acupuncture is fully covered" in statement.lower() and "excluded" in source.lower():
                return 0.0, {"reason": "Statement claims coverage, but policy explicitly lists acupuncture under exclusions."}
            return 1.0, {"reason": "All administrative statements are fully supported by policy/claim context."}
    provider = MockHealthcareProvider()

def evaluate_healthcare_groundedness(context: str, response: str):
    try:
        score, details = provider.groundedness_measure_with_cot_reasons(
            source=context,
            statement=response
        )
        reason = details.get("reason", "") if isinstance(details, dict) else str(details)
        return score, reason
    except Exception as e:
        return 0.0, f"Evaluation exception: {e}"

print("Healthcare Groundedness Metric successfully initialized.")

## 🧪 Running Evaluation across Healthcare Administrative Scenarios

We run the evaluator over four distinct administrative scenarios:
- **Supported**: Response accurately mirrors claim, coverage, or prior-auth facts.
- **Contradictory**: Response directly conflicts with policy limits or claim denial status.
- **Unsupported**: Response fabricates payment methods, check delivery dates, or unmentioned details.
- **Appropriate Abstention**: Assistant correctly refuses to approve pending authorizations or unverified requests.

In [ ]:
results = []

for case in dataset:
    score, reason = evaluate_healthcare_groundedness(case["context"], case["response"])
    is_correct = (abs(score - case["expected_groundedness"]) < 0.1)
    results.append({
        "id": case["id"],
        "category": case["category"],
        "scenario": case["scenario"],
        "query": case["query"],
        "expected_groundedness": case["expected_groundedness"],
        "predicted_groundedness": score,
        "eval_correct": is_correct,
        "cot_reason": reason
    })

df_results = pd.DataFrame(results)
print("Evaluation complete. Sample results:")
df_results[['id', 'category', 'scenario', 'expected_groundedness', 'predicted_groundedness', 'eval_correct', 'cot_reason']]

## 📊 Results Summary & Analysis

We calculate the overall evaluation accuracy and break down groundedness scores by administrative category.

In [ ]:
accuracy = df_results['eval_correct'].mean() * 100
print(f"=== Healthcare Groundedness Evaluation Accuracy: {accuracy:.1f}% ===
")

summary = df_results.groupby('category').agg(
    total_cases=('id', 'count'),
    eval_accuracy=('eval_correct', 'mean'),
    mean_expected_groundedness=('expected_groundedness', 'mean'),
    mean_predicted_groundedness=('predicted_groundedness', 'mean')
).reset_index()

print("Category Summary:")
print(summary.to_string(index=False))